In [2]:
# 单元格 1
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")

# --- 关键：确保项目根目录在路径中 ---
path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(path)

# --- 导入您的项目文件 ---
from utils.common_utils import printlog, set_seed, make_dir, calculate_and_save_metrics
from utils.common_params import *
from models.LD_Net import LD_Net

# --- 💡 关键修正：导入 AG_BiGRU 类 (替换了原来的 CNNRNNs) ---
from models.AG_BiGRU import AG_BiGRU

from trainTest.early_stopping.early_stopping import EarlyStopping
from trainTest.optimizers.get_optimizer import Optimizer
from trainTest.losses.get_loss import get_classification_loss
from trainTest.lr_schedulers.get_lr_scheduler import LrScheduler

# 设置随机种子
set_seed(seed=42)
print("所有依赖导入成功 (AG_BiGRU 准备就绪)。")

所有依赖导入成功 (AG_BiGRU 准备就绪)。


In [3]:
# 单元格 2
import os
import torch
from utils.common_utils import make_dir

# === 1. 数据集选择 ===
DATASET_NAME = "SIAT"

# --- 💡 关键修改：标记为 AG-BiGRU + Diff + Focal ---
MODEL_TYPE = 'AG_BiGRU_Diff_Focal'
# ------------------------------------------------

n_channels_check = 9

# === 2. 关键路径设置 (使用绝对路径) ===
LD_NET_MODEL_PATH = r"E:\基于肌电图的下肢运动模式识别\Code2\trainTest\LD_train Test\checkpoints\LD_Net_SIAT\ld_net_best_model.pth"
DATA_DIR = r"E:\基于肌电图的下肢运动模式识别\Code2\preProcessing\SIAT_LLMD_trainData\Denoising_TrainSet_XY"

# === 3. 模型与训练设置 ===
BATCH_SIZE = 64
LEARNING_RATE = 5e-4 # 保持您原来的 LR
EPOCHS = 200
PATIENCE = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 4. 检查参数 ===
print(f"--- 方案 4 (复刻版): AG-BiGRU + 差分特征 + Focal Loss ---")
print(f"设备: {DEVICE}")
print(f"LD-Net 路径: {LD_NET_MODEL_PATH}")

if not os.path.exists(LD_NET_MODEL_PATH):
    print(f"❌ 警告：找不到 LD-Net 模型文件！")
else:
    print(f"✅ LD-Net 模型文件确认存在。")

if C != n_channels_check:
    print(f"⚠️ 警告: common_params.py 中的 C={C} 与 SIAT (C=9) 不匹配!")
else:
    print("✅ C 参数检查通过。")

# === 5. 结果保存路径 (修改为增加创新点文件夹) ===
save_dir_root = r"D:\Users\王云通\Desktop\LD_Net_Online_Classification\增加创新点\AG_BiGRU_Diff_Focal"

save_dir_models_temp = save_dir_root
make_dir(save_dir_root)

print(f"✅ 最终结果将保存到: {save_dir_root}")

--- 方案 4 (复刻版): AG-BiGRU + 差分特征 + Focal Loss ---
设备: cuda
LD-Net 路径: E:\基于肌电图的下肢运动模式识别\Code2\trainTest\LD_train Test\checkpoints\LD_Net_SIAT\ld_net_best_model.pth
✅ LD-Net 模型文件确认存在。
✅ C 参数检查通过。
✅ 最终结果将保存到: D:\Users\王云通\Desktop\LD_Net_Online_Classification\增加创新点\AG_BiGRU_Diff_Focal


In [4]:
# 单元格 3
print("--- 正在加载 LD-Net 降噪模型 (冻结用于特征提取) ---")
try:
    model_denoise = torch.load(LD_NET_MODEL_PATH, map_location=DEVICE, weights_only=False)
    model_denoise.to(DEVICE)
    model_denoise.eval()

    # 额外加锁，确保不更新
    for param in model_denoise.parameters():
        param.requires_grad = False

    print("✅ LD-Net 降噪模型加载成功并设为 eval() 模式。")
except Exception as e:
    print(f"❌ 加载 LD-Net 模型失败: {e}")
    raise e

--- 正在加载 LD-Net 降噪模型 (冻结用于特征提取) ---
✅ LD-Net 降噪模型加载成功并设为 eval() 模式。


In [5]:
# 单元格 4
import glob

class LdNetClassificationDataset(Dataset):
    """
    用于分类器训练的数据集。
    从 步骤一 的 'Denoising_TrainSet_XY' 文件夹加载数据。
    """

    def __init__(self, data_dir, subject_id):
        self.data_dir = data_dir
        self.subject_id = subject_id

        # --- 匹配 'Denoising_TrainSet_XY' 的文件名 ---
        file_name_prefix = '_XY_TrainData.npz'
        search_path = os.path.join(self.data_dir, f"{self.subject_id}{file_name_prefix}")
        file_paths = glob.glob(search_path)

        if not file_paths:
            print(f"警告: 在 {self.data_dir} 中未找到 {self.subject_id}{file_name_prefix}。")
            self.samples = np.empty((0, C, window))
            self.labels_encoded = np.empty((0,))
            return

        try:
            data = np.load(file_paths[0])
            self.samples = data['noisy_X'].astype(np.float32)
            self.labels_encoded = data['sub_motion_label_encoded'].astype(np.int64)
            print(f"  已加载 {self.subject_id}: {self.samples.shape[0]} 个样本。")

        except Exception as e:
            print(f"加载 {file_paths[0]} 出错: {e}")
            self.samples = np.empty((0, C, window))
            self.labels_encoded = np.empty((0,))

    def __len__(self):
        return self.samples.shape[0]

    def __getitem__(self, idx):
        return self.samples[idx], self.labels_encoded[idx]

print("✅ 分类器数据集 (LdNetClassificationDataset) 定义完毕。")

✅ 分类器数据集 (LdNetClassificationDataset) 定义完毕。


In [6]:
# 单元格 5 (AG-BiGRU + Diff + Focal Loss)
from sklearn.model_selection import train_test_split
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.optim as optim
import numpy as np
import torch.nn as nn
from trainTest.metrics.metrics_utils import (
    get_accuracy, get_precision, get_recall, get_f1, get_specificity
)
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

# --- 1. 定义 Focal Loss (gamma=1.2) ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        else: return focal_loss

# --- 2. 辅助指标函数 ---
def _calc_cls_metrics(y_true, y_pred, y_scores=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    metrics = {
        'Accuracy': get_accuracy(y_true, y_pred), 'Precision': get_precision(y_true, y_pred),
        'Recall': get_recall(y_true, y_pred), 'F1': get_f1(y_true, y_pred),
        'Specificity': get_specificity(y_true, y_pred) ,
    }
    auc_pct = 0.0
    if y_scores is not None:
        if isinstance(y_scores, list):
            try: y_scores = np.asarray(y_scores)
            except Exception: y_scores = None
        if isinstance(y_scores, np.ndarray) and y_scores.ndim == 2 and y_scores.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_scores, multi_class='ovr', average='macro')
                auc_pct = round(auc * 100.0, 3)
            except Exception: auc_pct = 0.0
    metrics['AUC'] = auc_pct
    return metrics

# --- 3. 差分特征构造 (在线计算) ---
def _create_2ch_input(out, C_channels=9):
    if isinstance(out, (list, tuple)): t = out[-1] # 取最后输出
    else: t = out

    if not (t.dim() == 3 and t.size(1) == C_channels):
         if t.dim() == 4 and t.size(1) == 1: t = t.squeeze(1)
         else: raise RuntimeError(f"LD-Net (eval) 输出形状 {tuple(t.shape)} 异常")

    ch1_signal = t.unsqueeze(1)
    ch2_diff = t[..., 1:] - t[..., :-1]
    ch2_diff_padded = F.pad(ch2_diff, (1, 0), 'constant', 0)
    ch2_diff_signal = ch2_diff_padded.unsqueeze(1)

    return torch.cat([ch1_signal, ch2_diff_signal], dim=1) # -> [B, 2, 9, L]

# --- 4. 融合层 (TwoChanStem) ---
class TwoChanStem(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super().__init__()
        self.fuse = nn.Conv2d(in_channels, out_channels, kernel_size=(1,1), stride=(1,1), padding=0, bias=True)
        self.bn   = nn.BatchNorm2d(out_channels)
        self.act  = nn.GELU() # 升级为 GELU 配合 AG-BiGRU
    def forward(self, x):
        x = self.fuse(x); x = self.bn(x); return self.act(x)

# --- 5. 训练逻辑 ---
def train_one_epoch(model_classifier, twochan_stem,
                    model_denoiser, train_loader, criterion, optimizer, device):
    model_classifier.train(); twochan_stem.train(); model_denoiser.eval()
    train_loss = 0.; total_num = 0; y_true_all = []; y_pred_all = []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.no_grad():
            out = model_denoiser(inputs)
            x_2ch = _create_2ch_input(out, C_channels=C)

        inputs_denoised = twochan_stem(x_2ch)
        outputs = model_classifier(inputs_denoised)

        loss = criterion(outputs, labels) # Focal Loss

        optimizer.zero_grad(); loss.backward(); optimizer.step()

        bs = labels.size(0)
        train_loss += loss.item() * bs; total_num  += bs
        y_true_all.extend(labels.detach().cpu().numpy())
        y_pred_all.extend(torch.max(outputs, 1)[1].detach().cpu().numpy())

    return train_loss/total_num, _calc_cls_metrics(y_true_all, y_pred_all, None)

@torch.no_grad()
def test_one_epoch(model_classifier, twochan_stem,
                   model_denoiser, test_loader, criterion, device):
    model_classifier.eval(); twochan_stem.eval(); model_denoiser.eval()
    test_loss, total_num = 0.0, 0
    y_true_all, y_pred_all, y_scores_all = [], [], []

    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        out = model_denoiser(inputs)
        x_2ch = _create_2ch_input(out, C_channels=C)

        inputs_denoised = twochan_stem(x_2ch)
        outputs = model_classifier(inputs_denoised)

        loss = criterion(outputs, labels) # Focal Loss

        bs = labels.size(0)
        test_loss  += loss.item() * bs; total_num  += bs
        y_true_all.extend(labels.detach().cpu().numpy())
        y_pred_all.extend(torch.max(outputs, 1)[1].detach().cpu().numpy())
        y_scores_all.extend(F.softmax(outputs, dim=1).detach().cpu().numpy())

    return test_loss/total_num, _calc_cls_metrics(y_true_all, y_pred_all, np.asarray(y_scores_all)), y_true_all, y_pred_all

# --- 6. 主流程 ---
def train_test_subject(subject_id, data_dir, model_denoise, device, save_dir_models_temp,
                       model_type, epochs, batch_size, lr, patience, exp_tim):

    dataset = LdNetClassificationDataset(data_dir=data_dir, subject_id=subject_id)
    if len(dataset) == 0: return None
    train_idx, test_idx = train_test_split(
        list(range(len(dataset))), test_size=0.2, random_state=exp_tim, stratify=dataset.labels_encoded
    )

    train_loader = DataLoader(torch.utils.data.Subset(dataset, train_idx), batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(torch.utils.data.Subset(dataset, test_idx),  batch_size=batch_size, shuffle=False, num_workers=0)

    # 实例化模型
    twochan_stem = TwoChanStem().to(device)

    # --- 💡 关键修改：使用 AG_BiGRU ---
    from models.AG_BiGRU import AG_BiGRU
    model_classifier = AG_BiGRU(rnn_type='BiGRU').to(device)
    print("已实例化 1x1 Stem 适配器 和 AG-BiGRU 分类器。")

    # 使用 Focal Loss (gamma=1.2)
    criterion = FocalLoss(gamma=1.2).to(device)
    print("  > 使用 FocalLoss (gamma=1.2) 作为损失函数。")

    # 优化器设置
    print("正在为 Stem 和 分类器 设置优化器 (LD-Net 已冻结)...")
    stem_lr = lr * 2.0
    classifier_lr = lr

    optimizer = optim.AdamW([
        {'params': twochan_stem.parameters(), 'lr': stem_lr},
        {'params': model_classifier.parameters(), 'lr': classifier_lr}
    ], lr=lr, weight_decay=1e-4)

    # 调度器
    scheduler_params = {'ReduceLROnPlateau': {'mode':'min','factor':0.1,'patience':5,'threshold':1e-4,'min_lr':1e-8}}
    scheduler = LrScheduler(optimizer=optimizer, scheduler_type='ReduceLROnPlateau',
                            params=scheduler_params, max_epoch=epochs).get_scheduler()

    save_dir_subj = os.path.join(save_dir_models_temp, subject_id)
    make_dir(save_dir_subj)
    best_loss_path = os.path.join(save_dir_subj, f'best_loss_model_exp{exp_tim}.pth')
    early_stopping  = EarlyStopping(patience=patience, verbose=True, path=best_loss_path)
    best_acc = -1.0
    best_acc_path = os.path.join(save_dir_subj, f'best_acc_model_exp{exp_tim}.pth')

    def save_models_checkpoint(path):
        torch.save({
            'twochan_stem_state_dict': twochan_stem.state_dict(),
            'model_classifier_state_dict': model_classifier.state_dict(),
        }, path)

    # 训练循环
    for epoch in range(1, epochs+1):
        train_loss, train_metrics = train_one_epoch(
            model_classifier, twochan_stem, model_denoise,
            train_loader, criterion, optimizer, device
        )
        val_loss, val_metrics, _, _ = test_one_epoch(
            model_classifier, twochan_stem, model_denoise,
            test_loader, criterion, device
        )
        scheduler.step(val_loss)

        state_dict = {
            'twochan_stem_state_dict': twochan_stem.state_dict(),
            'model_classifier_state_dict': model_classifier.state_dict(),
        }
        early_stopping(val_loss, state_dict)

        val_acc = val_metrics['Accuracy']
        if val_acc > best_acc:
            best_acc = val_acc
            save_models_checkpoint(best_acc_path)
            print(f"    {subject_id} | Exp {exp_tim} | 新的最佳准确率: {val_acc:.2f}% (已保存)")

        if (epoch % 10 == 0) or early_stopping.early_stop:
            print(f"{subject_id} | Exp {exp_tim} | Ep {epoch:03d} | "
                  f"Train {train_loss:.4f}/{train_metrics['Accuracy']:.2f}% | "
                  f"Val {val_loss:.4f}/{val_metrics['Accuracy']:.2f}%")
        if early_stopping.early_stop:
            print("早停触发。"); break

    # 最终测试
    load_path = best_acc_path if os.path.exists(best_acc_path) else best_loss_path
    print(f"加载 {subject_id} (Exp {exp_tim}) 的最佳模型 (来自: {load_path})...")

    # 重新实例化以防万一
    best_stem = TwoChanStem().to(device)
    best_classifier = AG_BiGRU(rnn_type='BiGRU').to(device)

    ckpt = torch.load(load_path)
    best_stem.load_state_dict(ckpt['twochan_stem_state_dict'])
    best_classifier.load_state_dict(ckpt['model_classifier_state_dict'])

    test_loss_final, test_metrics_final, y_true_final, y_pred_final = test_one_epoch(
        best_classifier, best_stem, model_denoise,
        test_loader, criterion, device
    )

    # 混淆矩阵
    motions_list_cm = ['WAK', 'STDUP', 'SITDN', 'UPS', 'DNS']
    cm_save_jpg_name = os.path.join(save_dir_subj, f'confusion_matrix_exp{exp_tim}.jpg')
    cm_save_csv_name = os.path.join(save_dir_subj, f'confusion_matrix_exp{exp_tim}.xlsx')
    PlotConfusionMatrix(
        y_true_final, y_pred_final, label_type=motions_list_cm,
        show_type='all', plot=False, save_fig=True, save_results=True, cmap='YlGnBu'
    ).get_confusion_matrix(cm_save_jpg_name, cm_save_csv_name)

    test_metrics_final['Subject'] = subject_id
    return test_metrics_final

print("✅ 训练/测试函数 (train_test_subject) [AG-BiGRU + Diff + Focal(1.2)] 定义完毕。")

✅ 训练/测试函数 (train_test_subject) [AG-BiGRU + Diff + Focal(1.2)] 定义完毕。


In [7]:
# 单元格 6
import pandas as pd
import numpy as np

# 确保 K 存在
if 'K_of_repeated_experiments' not in globals():
    K_of_repeated_experiments = 5

subjects_list = ['Sub01', 'Sub02', 'Sub03', 'Sub04', 'Sub05', 'Sub31', 'Sub32', 'Sub33', 'Sub34', 'Sub35']
# subjects_list = ['Sub02'] # 调试用

metrics_all_subjects = []

print(f"--- 开始在 {DATASET_NAME} 上执行 {MODEL_TYPE} 受试者内测试 ---")
print(f"--- 结果保存至: {save_dir_root} ---")

for subject in subjects_list:
    for exp_tim in range(1, K_of_repeated_experiments + 1):

        print(f"\n>>> 处理受试者: {subject} | 实验: {exp_tim}/{K_of_repeated_experiments} <<<")

        test_metrics = train_test_subject(
            subject_id=subject,
            data_dir=DATA_DIR,
            model_denoise=model_denoise,
            device=DEVICE,
            save_dir_models_temp=save_dir_models_temp,
            model_type=MODEL_TYPE,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            patience=PATIENCE,
            exp_tim=exp_tim
        )

        if test_metrics:
            test_metrics['Experiment'] = exp_tim
            metrics_all_subjects.append(test_metrics)
            print(f"--- {subject} Exp {exp_tim} 完成。Acc: {test_metrics['Accuracy']:.2f}% ---")

print("\n--- 所有受试者处理完毕 ---")

# 生成统计表格
if metrics_all_subjects:
    df_summary = pd.DataFrame(metrics_all_subjects)
    save_path_summary_by_subject = os.path.join(save_dir_root, 'summary_by_subject.csv')
    df_summary.to_csv(save_path_summary_by_subject, index=False, float_format='%.3f')
    print(f"✅ 详细数据已保存: {save_path_summary_by_subject}")

    df_avg = df_summary.groupby('Subject').mean(numeric_only=True)
    df_std = df_summary.groupby('Subject').std(numeric_only=True)

    df_formatted = pd.DataFrame(index=df_avg.index)
    metrics_cols = [c for c in df_avg.columns if c != 'Experiment']

    for col in metrics_cols:
        df_formatted[col] = df_avg[col].map('{:.2f}'.format) + " ± " + df_std[col].map('{:.2f}'.format)

    df_total_avg = df_avg[metrics_cols].mean().to_frame().T
    df_total_std = df_avg[metrics_cols].std().to_frame().T

    df_total_avg_fmt = df_total_avg.applymap(lambda x: f"{x:.2f}")
    df_total_std_fmt = df_total_std.applymap(lambda x: f"{x:.2f}")
    df_total_avg_fmt.index = ['Mean']
    df_total_std_fmt.index = ['Std']

    df_final = pd.concat([df_formatted, df_total_avg_fmt, df_total_std_fmt])

    save_path_final = os.path.join(save_dir_root, 'all_metrics_averaged_results.csv')
    df_final.to_csv(save_path_final, index=True, encoding='utf-8-sig')
    print(f"✅ 最终报告已保存: {save_path_final}")

    try:
        sub02 = df_avg.loc['Sub02']['Accuracy']
        print(f"\n--- Sub02 最终 Acc: {sub02:.2f}% ---")
    except: pass
else:
    print("没有生成数据。")

--- 开始在 SIAT 上执行 AG_BiGRU_Diff_Focal 受试者内测试 ---
--- 结果保存至: D:\Users\王云通\Desktop\LD_Net_Online_Classification\增加创新点\AG_BiGRU_Diff_Focal ---

>>> 处理受试者: Sub01 | 实验: 1/5 <<<
  已加载 Sub01: 690 个样本。
已实例化 1x1 Stem 适配器 和 AG-BiGRU 分类器。
  > 使用 FocalLoss (gamma=1.2) 作为损失函数。
正在为 Stem 和 分类器 设置优化器 (LD-Net 已冻结)...
Validation loss decreased (inf --> 1.229725).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 29.71% (已保存)
Validation loss decreased (1.229725 --> 1.226541).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 30.43% (已保存)
Validation loss decreased (1.226541 --> 1.224840).  Saving model ...
Validation loss decreased (1.224840 --> 1.218802).  Saving model ...
Validation loss decreased (1.218802 --> 1.189517).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 41.30% (已保存)
Validation loss decreased (1.189517 --> 1.120821).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 52.90% (已保存)
Validation loss decreased (1.120821 --> 0.968602).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 55.80% (已保存)
Validation 